# 01 — Simulation demo

Demonstrates the `startrail` simulator:
- Single star trail (PSF convolved with line kernel)
- Effect of jitter on trail morphology
- Multi-star simulated CCD image with sky and read noise

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from astropy.table import Table

from startrail.simulate import make_image, make_trail, gaussian_psf

## 1. PSF models

In [ ]:
from startrail.simulate import gaussian_psf, moffat_psf

psf_g = gaussian_psf(31, fwhm=3.0)
psf_m = moffat_psf(31, fwhm=3.0, beta=4.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, psf, label in zip(axes, [psf_g, psf_m], ['Gaussian', 'Moffat β=4']):
    im = ax.imshow(psf, origin='lower', cmap='inferno')
    ax.set_title(label)
    plt.colorbar(im, ax=ax)
plt.suptitle('PSF models (FWHM=3 px)', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/01_psf_models.png', dpi=120)
plt.show()

## 2. Single trail — effect of trail length

In [ ]:
SHAPE = (64, 64)
cx, cy = 32.0, 32.0
fwhm = 3.0
flux = 10000.0

configs = [
    ('length=5',  5),
    ('length=20', 20),
    ('length=50', 50),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (label, length) in zip(axes, configs):
    trail = make_trail(SHAPE, cx, cy, angle=0.0, length=length, fwhm=fwhm, flux=flux)
    ax.imshow(trail, origin='lower', cmap='hot')
    ax.set_title(label, fontsize=10)
    ax.axis('off')
plt.suptitle('Trail morphology — readout direction (angle=0)', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/01_trail_morphology.png', dpi=120)
plt.show()

## 3. Effect of jitter

In [ ]:
jitter_vals = [0.0, 0.5, 1.5]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, jitter in zip(axes, jitter_vals):
    trail = make_trail(SHAPE, cx, cy, angle=0.0, length=20.0, fwhm=fwhm,
                       flux=flux, jitter_sigma=jitter, seed=0)
    ax.imshow(trail, origin='lower', cmap='hot')
    ax.set_title(f'jitter_sigma={jitter}', fontsize=10)
    ax.axis('off')
plt.suptitle('Jitter effect on trail morphology', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/01_jitter_effect.png', dpi=120)
plt.show()

## 4. Full simulated CCD image

In [ ]:
rng = np.random.default_rng(99)
n_stars = 12
IMAGE_SHAPE = (512, 512)
TRAIL_LENGTH = 20.0   # same for every star: set by the CCD readout time

stars = Table({
    'x':      rng.uniform(50, 462, n_stars),
    'y':      rng.uniform(50, 462, n_stars),
    'angle':  np.zeros(n_stars),           # readout direction
    'length': np.full(n_stars, TRAIL_LENGTH),
    'flux':   10 ** rng.uniform(3.5, 5.5, n_stars),
})

image = make_image(
    IMAGE_SHAPE, stars, fwhm=3.0,
    sky_level=200.0, read_noise=8.0, bias=1000.0,
    jitter_sigma=0.3, tau=50.0, seed=42,
)

fig, ax = plt.subplots(figsize=(8, 8))
vmin, vmax = np.percentile(image, [1, 99.5])
ax.imshow(image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
ax.set_title(
    f'Simulated CCD image — {n_stars} stars, trail length={TRAIL_LENGTH:.0f} px (fixed)',
    fontsize=12,
)
plt.tight_layout()
plt.savefig('../plots/01_simulated_image.png', dpi=120)
plt.show()

print(f'Image shape: {image.shape}')
print(f'Min / max: {image.min():.1f} / {image.max():.1f}')
print(f'Mean sky + bias: {image.mean():.1f}')

## 5. Long jittered trail (>1000 rows)

A single star spanning most of a tall CCD readout, comparing a perfect trail to one
with realistic telescope tracking jitter.

In [ ]:
LONG_SHAPE = (1200, 120)   # 1200-row CCD readout
cx_long = 60.0             # star centred horizontally
cy_long = 600.0            # star centred vertically
length_long = 1100.0       # trail spans 1100 of the 1200 rows
fwhm_long = 4.0
flux_long = 5e5
TAU = 150.0                # guider e-folding timescale in pixels

trail_straight = make_trail(
    LONG_SHAPE, cx_long, cy_long,
    angle=0.0, length=length_long, fwhm=fwhm_long, flux=flux_long,
)
trail_jittered = make_trail(
    LONG_SHAPE, cx_long, cy_long,
    angle=0.0, length=length_long, fwhm=fwhm_long, flux=flux_long,
    jitter_sigma=1.5, tau=TAU, seed=7,
)

fig, axes = plt.subplots(1, 2, figsize=(8, 12))
for ax, trail, label in zip(
    axes,
    [trail_straight, trail_jittered],
    ['No jitter', f'OU jitter  σ=1.5 px,  τ={TAU:.0f} px'],
):
    vmax = np.percentile(trail[trail > 0], 99)
    ax.imshow(trail, origin='lower', cmap='hot', vmin=0, vmax=vmax, aspect='auto')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('column (px)')
    ax.set_ylabel('row (px)')

plt.suptitle('1100-row open-shutter trail (1200-row CCD)\nOU jitter: guider restores star on τ-pixel timescale', fontsize=12)
plt.tight_layout()
plt.savefig('../plots/01_long_jittered_trail.png', dpi=120)
plt.show()

# Show the jitter displacement as a function of row
n_steps = max(int(length_long * 3), 30)
rng = np.random.default_rng(7)
dt = length_long / n_steps
decay = np.exp(-dt / TAU)
drive = 1.5 * np.sqrt(1.0 - decay**2)
j = np.zeros(n_steps)
for i in range(1, n_steps):
    j[i] = decay * j[i-1] + drive * rng.normal()

rows = np.linspace(cy_long - length_long/2, cy_long + length_long/2, n_steps)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(rows, j, lw=0.8)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.axhline( 1.5, color='r', lw=0.5, ls=':', label='±1σ')
ax.axhline(-1.5, color='r', lw=0.5, ls=':')
ax.set_xlabel('row (px)')
ax.set_ylabel('jitter displacement (px)')
ax.set_title(f'Jitter track — OU process  σ=1.5 px,  τ={TAU:.0f} px')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/01_jitter_track.png', dpi=120)
plt.show()

print(f'Jitter RMS: {j.std():.2f} px  (target σ=1.5 px)')
print(f'Jitter range: [{j.min():.2f}, {j.max():.2f}] px')